# ODI to Databricks Migration: TRG_EMP_INSERT

Conversion Timestamp: 2024-07-30 12:00:00

This notebook converts an ODI `INSERT` statement from Oracle to Databricks Spark SQL. It loads data from the `EMPLOYEES` table into the `TRG_EMP` table.

In [ ]:
import dbutils

dbutils.widgets.text("ETL_JOB_TYPE", "FULL_LOAD", "1. ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "100", "3. ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "12345", "4. ODI Session Number")

# ETL Parameters

The following parameters control the execution of this notebook. Default values are provided but can be overridden when scheduling.

In [ ]:
display(spark.sql("""
  SELECT
    '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,
    ${DATASOURCE_NUM_ID} AS DATASOURCE_NUM_ID,
    ${ETL_PROC_WID} AS ETL_PROC_WID,
    '${ODI_SESS_NO}' AS ODI_SESS_NO
"""))

# Insert Data into Target Table

This step inserts records from the source `EMPLOYEES` table into the target `TRG_EMP` table.

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}
INSERT INTO workspace.hr.trg_emp (
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID
)
SELECT
    EMPLOYEES.EMPLOYEE_ID,
    EMPLOYEES.FIRST_NAME,
    EMPLOYEES.LAST_NAME,
    EMPLOYEES.EMAIL,
    EMPLOYEES.PHONE_NUMBER,
    EMPLOYEES.HIRE_DATE,
    EMPLOYEES.JOB_ID,
    EMPLOYEES.SALARY,
    EMPLOYEES.COMMISSION_PCT,
    EMPLOYEES.MANAGER_ID,
    EMPLOYEES.DEPARTMENT_ID
FROM
    workspace.hr.employees AS EMPLOYEES;

# Validation

Verify the number of records in the target table after the insert.

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

# Conversion Notes

1.  **Schema and Table Names**: Oracle schema `HR` has been converted to `workspace.hr`. Table names `TRG_EMP` and `EMPLOYEES` have been converted to lowercase: `trg_emp` and `employees`.
2.  **Oracle Hints**: The Oracle hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable in Databricks Spark SQL.
3.  **SCEN_TASK_NO Markers**: `SCEN_TASK_NO {10}` and `{20}` were empty in the source and are included as comments in the subsequent SQL cell.
4.  **Widgets**: Standard ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been added as Databricks widgets for consistency, as per conversion guidelines, although they are not directly used in this specific `INSERT` statement.